# 📨 CHRUTH — Messages par appel d'offres (clé API ou Ollama)

Choisis un AO **intéressant** (CHAUD/TIÈDE), l'IA rédige un **email structuré + un script d'appel** à l'acheteur, prêts à copier/coller et modifier.

**Moteur automatique :** si une **clé cloud** est dans `.env` → cloud (rapide, structuré) ; sinon **Ollama** local s'il tourne ; sinon **brouillon déterministe**.

**Précision :** remplis `config_chruth/fiche_chruth.md` avec les vrais faits CHRUTH — l'IA n'utilisera que ça.

**Étapes :** 1) Setup — 2) repère le n° de l'AO — 3) mets-le dans `CHOIX` — 4) Générer. Le brouillon est aussi écrit dans `output/messages_ao/AO_<id>.md` (éditable).

## Setup

In [1]:
import sys, pathlib, importlib
sys.path.insert(0, str(pathlib.Path.cwd()))
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
import llm_client, prospect_messages as pm, ao_messages as am
from ao_db import connect
from ao_config import AO_DB_PATH
importlib.reload(llm_client); importlib.reload(pm); importlib.reload(am)

FICHE = pm.fiche_chruth()
_moteur = llm_client.moteur_auto()
print('Moteur IA :', _moteur or 'aucun -> brouillon deterministe')
print('Fiche CHRUTH :', 'chargee' if FICHE else 'VIDE (remplis config_chruth/fiche_chruth.md)')

with connect(AO_DB_PATH) as _c:
    AOS = [dict(r) for r in _c.execute(
        "SELECT * FROM ao_records WHERE priorite IN ('CHAUD','TIEDE') "
        "ORDER BY CAST(score_chruth AS INTEGER) DESC").fetchall()]
print(len(AOS), 'AO CHAUD/TIEDE disponibles')

Moteur IA : ollama
Fiche CHRUTH : VIDE (remplis config_chruth/fiche_chruth.md)
45 AO CHAUD/TIEDE disponibles


## 1. Liste des AO — repère le numéro voulu

In [2]:
for i, a in enumerate(AOS):
    print(f"[{i:>3}] {str(a.get('priorite','')):6} | "
          f"{str(a.get('objet',''))[:70]:70} | {a.get('acheteur','')}")

[  0] CHAUD  | Accord cadre de missions de formation de lutte contre la discriminatio | VILLE de PARIS - DFA - SDA
[  1] CHAUD  | Nettoyage des vitres et entretien des bâtiments communaux (Administrat | MAIRIE DES LILAS
[  2] CHAUD  | FOURNITURE D'EQUIPEMENTS, PRODUITS D'ENTRETIEN ET HACCP AU PROFIT DE L | Ville de Longjumeau
[  3] CHAUD  | Restauration de monuments funéraires et de statues pour la ville d'Asn | Mairie d'Asnières sur Seine
[  4] CHAUD  | Prestations de nettoyage de certains équipements de protection individ | MAIRIE DE ROMAINVILLE
[  5] CHAUD  | Prestations de bio-nettoyage, mise à blanc et mise à gris des zones à  | DAPSA/PFAF-S
[  6] CHAUD  | Accord-cadre relatif à la maintenance préventive et corrective des air | Mairie de Mantes La Jolie
[  7] CHAUD  | Prestations de nettoyage courant des locaux administratifs de Pantin H | Pantin Habitat
[  8] CHAUD  | Nettoyage des locaux des bâtiments de la commune de Châtillon (92320)  | Commune de CHATILLON
[  9] CHAUD  | PRES

## 2. Choisis un AO

In [3]:
CHOIX = 0   # numero de l'AO dans la liste ci-dessus

## ▶️ Générer — message structuré prêt à copier/coller

In [4]:
ao = AOS[CHOIX]
msg = am.generer_message_ao(ao, fiche=FICHE)
chemin = am.ecrire_brouillon_md(ao, msg)
print('AO       :', ao.get('objet', ''))
print('Acheteur :', ao.get('acheteur', ''), '| Ville :', ao.get('ville', ''),
      '| Date limite :', ao.get('date_limite', ''))
print('Source   :', msg['source'], '(ia = redige par le modele ; defaut = brouillon type)')
print('Fichier editable :', chemin)
print('\n' + '=' * 72 + '\nEMAIL\n' + '=' * 72)
print(msg['email'])
print('\n' + '=' * 72 + "\nSCRIPT D'APPEL\n" + '=' * 72)
print(msg['script'])

AO       : Accord cadre de missions de formation de lutte contre la discrimination et de lutte contre les violences sexistes et sexuelles à destination du tissu associatif en deux lots séparés.
Acheteur : VILLE de PARIS - DFA - SDA | Ville :  | Date limite : 2026-09-15T10:00:00+00:00
Source   : defaut (ia = redige par le modele ; defaut = brouillon type)
Fichier editable : %USERPROFILE%\Downloads\CHRUTH_LIVRAISON_NO_CODE\output\messages_ao\AO_26-63741.md

EMAIL
Bonjour,

Je me permets de vous contacter au sujet de votre appel d'offres : Accord cadre de missions de formation de lutte contre la discrimination et de lutte contre les violences sexistes et sexuelles à destination du tissu associatif en deux lots séparés..

CHRUTH accompagne les structures publiques et privées sur les besoins de nettoyage, d'entretien des locaux et d'organisation de prestations de service.

Serait-il possible d'échanger rapidement afin de vérifier les modalités de réponse et les attentes de VILLE de PARIS - 

## Astuces
- Change `CHOIX` et relance *Générer* pour un autre AO.
- **Plus de précision** : édite `config_chruth/fiche_chruth.md` (faits réels CHRUTH).
- **Style** : édite `prompt_ao()` dans `ao_messages.py`.
- Le brouillon éditable est dans `output/messages_ao/`, dans le cockpit `output/AO_CHRUTH.xlsm`, et injecté dans le **mail d'alerte** AO.